# LlamaIndex + BuyWhere MCP: Product Search Agent

This notebook demonstrates how to use the [BuyWhere MCP server](https://github.com/BuyWhere/buywhere-mcp) with LlamaIndex to build an AI agent that can search and compare products across Singapore, SEA, and US e-commerce platforms.

BuyWhere provides access to 3M+ products across Shopee, Lazada, Amazon, Walmart, FairPrice, Carousell, and 20+ other platforms.

We use the `llama-index-tools-mcp` package to connect to the BuyWhere MCP server via stdio transport.

In [ ]:
%pip install llama-index-tools-mcp llama-index-llms-openai llama-index-core

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-..."  # replace with your OpenAI API key

## Load BuyWhere MCP Tools

The BuyWhere MCP server runs via `npx @buywhere/mcp-server` (requires Node.js 18+). LlamaIndex connects to it via stdio transport using `get_tools_from_mcp_url`.

In [ ]:
from llama_index.tools.mcp import get_tools_from_mcp_url

# Connect to BuyWhere MCP server via stdio
# The server is launched automatically via npx
tools = await get_tools_from_mcp_url(
    "stdio://npx @buywhere/mcp-server"
)

print(f"Loaded {len(tools)} tools from BuyWhere MCP server:")
for tool in tools:
    print(f"  - {tool.metadata.name}: {tool.metadata.description[:80]}")

## Build a Product Search Agent

Now we create a LlamaIndex `ReActAgent` that uses the BuyWhere tools to answer product search queries.

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini")

agent = ReActAgent.from_tools(
    tools,
    llm=llm,
    verbose=True,
    max_iterations=10,
)

## Example Queries

### Search for products across Singapore platforms

In [ ]:
response = await agent.achat(
    "Find me the best deals on air purifiers in Singapore under SGD 300. "
    "Compare prices across Shopee, Lazada, and FairPrice."
)
print(response)

### Cross-border price comparison

In [ ]:
response = await agent.achat(
    "I want to buy a MacBook Pro M3. Compare prices between Singapore "
    "and US Amazon, and tell me where it's cheaper after accounting for "
    "typical import duties."
)
print(response)

## Resources

- [BuyWhere MCP GitHub](https://github.com/BuyWhere/buywhere-mcp)
- [BuyWhere API docs](https://buywhere.ai)
- [llama-index-tools-mcp docs](https://docs.llamaindex.ai/en/stable/examples/tools/mcp/)
